# Shotgun metagenomics MAG recovery



In [1]:
import pandas as pd
import qiime2 as q2
import seaborn as sns

import matplotlib.pyplot as plt
%matplotlib inline

## 1. Genome assembly

The first step in recovering metagenome-assembled genomes (MAGs) is genome assembly itself. There are many genome assemblers available, two of which you can use through our QIIME 2 plugin - here, we will use MEGAHIT. MEGAHIT takes short DNA sequencing reads, constructs a simplified De Bruijn graph, and generates longer contiguous sequences called contigs, providing valuable genetic information for the next steps of our analysis.

Coffee break!

In [ ]:
! qiime assembly assemble-megahit \
    --i-reads  /data/$data_dir/reads.qza \
    --p-presets meta-sensitive \
    --p-num-cpu-threads 4 \
    --o-contigs $data_dir/contigs.qza

### Quality assessment
Once the reads are assembled into contigs, we can use QUAST to evaluate the quality of our assembly. There are many metrics which can be used for that purpose but here we will focus on the two most popular metrics:

N50: represents the contiguity of a genome assembly. It's defined as the length of the contig (or scaffold) at which 50% of the entire genome is covered by contigs of that length or longer - the higher this number, the better.

L50: represents the number of contigs required to cover 50% of the genome's total length - the smaller this number, the better.



In [ ]:
! qiime assembly evaluate-quast \
    --i-contigs $data_dir/contigs.qza \
    --i-references /data/$data_dir/ref-seqs.qza \
    --p-min-contig 500 \
    --p-threads 4 \
    --o-reference-genomes $data_dir/ref-genomes.qza \ 
    --o-visualization $data_dir/contigs.qzv \
    --o-results-table $data_dir/quast-results.qza

In [ ]:
q2.Visualization.load(f"{data_dir}/contigs.qzv")

## 2. Read mapping
Before we continue to assembling MAGs, we need to index the contigs obtained in the assembly step and map the original reads to those contigs using that index. This read mapping can then be used by the contig binner to figure out which contigs originated from the same genome and put those together. Execute the two cells below to run the contig indexing and read mapping actions (that should take only a few minutes):  

In [ ]:
! qiime assembly index-contigs \
    --i-contigs $data_dir/contigs.qza \
    --p-threads 4 \
    --p-seed 100 \
    --o-index $data_dir/contigs-index.qza

In [ ]:
! qiime assembly map-reads \
    --i-index $data_dir/contigs-index.qza \
    --i-reads /data/$data_dir/reads.qza \
    --p-threads 4 \
    --p-seed 100 \
    --o-alignment-maps $data_dir/reads-to-contigs-aln.qza

## 3. Contig binning

Finally, we are ready to perform contig binning. This process involves categorizing contigs into distinct bins or groups based on their likely origin from different microbial species or strains within a mixed community. Here, we will use the MetaBAT 2 tool, which uses tetranucleotide frequency together with abundance (coverage) information to assign contigs to individual bins.

In [ ]:
! qiime mag bin-contigs-metabat \
    --i-contigs $data_dir/contigs.qza \
    --i-alignment-maps $data_dir/reads-to-contigs-aln.qza \
    --p-num-threads 4 \
    --p-seed 100 \
    --o-mags $data_dir/mags.qza \
    --o-contig-map $data_dir/contig-map.qza \
    --o-unbinned-contigs $data_dir/unbinned-contigs.qza

## 4. MAG quality control

The previous steps generated a couple artifacts:

mags.qza: these are our actual MAGS, per sample
contig-map.qza: this is a mapping between MAG IDs and IDs of contigs which belong to a given MAG
unbinned-contigs.qza: these are all the contigs that could not be assign to any particular MAG
Once we have our contigs binned into Metagenome-Assembled Genomes (MAGs), we need to check what the quality of those bins is. There are a couple of diffeernt tools which can be used for this purpose, many of which use the single-copy marker genes to estimate the completeness and purity (or contamination) of the recovered genomes. Here, we will use BUSCO which uses a set of curated ortholog genes to estimate those metrics.

We begin by fetching the required BUSCO database: we know that all species in our samples are strictly bacteria so we can fetch only this lineage to save some space and resources:

In [ ]:
! qiime mag fetch-busco-db \
    --p-lineages bacteria_odb12 \
    --o-db $data_dir/busco-db-bacteria.qza

In [ ]:
! qiime mag evaluate-busco \
    --i-mags $data_dir/mags.qza \
    --i-db $data_dir/busco-db-bacteria.qza \
    --i-unbinned-contigs $data_dir/unbinned-contigs.qza \
    --p-lineage-dataset bacteria_odb12 \
    --p-cpu 4 \
    --o-results $data_dir/busco-results.qza \
    --o-visualization $data_dir/mags.qzv

In [ ]:
q2.Visualization.load(f"{data_dir}/mags.qzv")

Now that we evaluated the quality of our MAGs, we can use this information to filter out only the best ones. We do not want to continue with MAGs of low quality (not very complete or highly contaminated) as it may lead to incorrect results in the downstream analyses. We want to keep the MAGs which are at least 50% complete and have less than 10% contamination (these are considered to be of medium quality according to the MIMAG standard). We can easily achieve this with the following action:

In [ ]:
! qiime mag filter-mags \
    --i-mags $data_dir/mags.qza \
    --m-metadata-file $data_dir/busco-results.qza \
    --p-where "completeness>50 AND contamination<10" \
    --p-on "mag" \
    --o-filtered-mags $data_dir/mags-filtered.qza

## 5. MAG dereplication

Since our samples were generated from the same mock community (i.e., we know they most likely contain the same set of genomes), we can simplify our MAG collection by performing dereplication, similarly to how you do it for 16S amplicon sequences (remember previous week?). To dereplicate our MAGs, we will:

compute hash sketches of every genome using sourmash - you can think of those sketches as tiny representations of our genomes (sourmash compresses a lot of information into much smaller space)
compare all of those sketches (genomes) to one another to generate a matrix of pairwise distances between our MAGs
dereplicate the genomes using the distance matrix and a fixed similarity threshold: the last action will simply choose the most complete genome from all of the genomes belonging to the same cluster, given a similarity threshold

In [ ]:
! qiime sourmash compute \
    --i-sequence-file $data_dir/mags-filtered.qza \
    --p-ksizes 105 \
    --p-scaled 100 \
    --o-min-hash-signature $data_dir/mags-hashes.qza

In [ ]:
! qiime sourmash compare \
    --i-min-hash-signature $data_dir/mags-hashes.qza \
    --p-ksize 105 \
    --o-compare-output $data_dir/mags-dist.qza

In [ ]:
! qiime mag dereplicate-mags \
    --i-mags $data_dir/mags-filtered.qza \
    --i-distance-matrix $data_dir/mags-dist.qza \
    --m-metadata-file $data_dir/busco-results.qza \
    --p-metadata-column "completeness" \
    --p-threshold 0.7 \
    --o-dereplicated-mags $data_dir/mags-derep.qza \
    --o-table $data_dir/mags-table-pa.qza

In [ ]:
pa_table = q2.Artifact.load(f"{data_dir}/mags-table-pa.qza").view(pd.DataFrame)

In [ ]:
sns.heatmap(pa_table.T, cmap='coolwarm', linewidths=0.5);